# VibeML: Personalized Spotify Playlist Agent

This notebook defines the VibeML agent, a personalized Spotify playlist generation agent that combines a user's listening history with real-time mood and activity input to build a playlist for the current moment.

##Install Dependencies

In [0]:
# Install dependencies
%pip install spotipy

In [0]:
%restart_python

## Imports and Setup

Import all necessary libraries and set up the environment for the agent.

In [0]:
# Core libraries
import os
import json
import pandas as pd

# Spotify API
import spotipy
from spotipy.oauth2 import SpotifyOAuth

# Databricks / MLflow
import mlflow

# LLM - we will fill this in once we confirm which LLM we are using

### Spotify Authentication

This section sets up the Spotify client using the full OAuth flow, which gives the agent access to user specific data including liked songs, playlists, listening history, and the ability to save playlists.

#### Setup Steps (first time only)

1. Go to developer.spotify.com and log in with your Spotify account
2. Click "Create app" and fill in the following:
   - App name: VibeML
   - App description: Personalized Spotify Playlist Agent
   - Redirect URI: http://127.0.0.1:8080
   - Check the "Web API" box
3. Once the app is created click "Settings" and copy your Client ID and Client Secret
4. Run the credentials cell below and enter your Client ID and Client Secret when prompted
5. Run this cell, it will print a Spotify authorization URL in the output:
    e.g. with the client ID and key as your specific credentials : "https://accounts.spotify.com/authorize?client_id=e883018d172a497a94c2e78f9b300117&response_type=code&redirect_uri=http%3A%2F%2F127.0.0.1%3A8080&scope=user-library-read+user-read-recently-played+playlist-read-private+playlist-modify-public+playlist-modify-private+user-read-currently-playing"
    
6. Copy that URL and paste it into your browser
7. Log in with your Spotify account and click "Agree"
8. You will be redirected to a URL starting with http://127.0.0.1:8080/?code=
9. Copy that full URL from your browser address bar
10. Paste it into the notebook where it says "Enter the URL you were redirected to:" and hit enter
11. You should see "Spotify connected successfully" with your display name

In [0]:
# CREDENTIALS CELL
import os

# Set Spotify credentials manually at runtime
# These are never stored in the notebook or GitHub
os.environ["SPOTIFY_CLIENT_ID"] = dbutils.widgets.get("client_id") if False else input("Enter your Spotify Client ID: ")
os.environ["SPOTIFY_CLIENT_SECRET"] = dbutils.widgets.get("client_secret") if False else input("Enter your Spotify Client Secret: ")
os.environ["SPOTIFY_REDIRECT_URI"] = "http://127.0.0.1:8080"

In [0]:
# AUTHENTICATION CELL
from spotipy.oauth2 import SpotifyClientCredentials

# Set up the Spotify client using client credentials
sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=os.environ["SPOTIFY_CLIENT_ID"],
    client_secret=os.environ["SPOTIFY_CLIENT_SECRET"]
))

In [0]:
# Test the connection
results = sp.search(q="test", limit=1)
print("Spotify connected successfully")
print(f"Test track: {results['tracks']['items'][0]['name']}")

In [0]:
from spotipy.oauth2 import SpotifyOAuth

# Define the scopes we need from the Spotify API
SPOTIFY_SCOPES = " ".join([
    "user-library-read",
    "user-read-recently-played",
    "playlist-read-private",
    "playlist-modify-public",
    "playlist-modify-private",
    "user-read-currently-playing"
])

# Set up the Spotify OAuth client
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=os.environ["SPOTIFY_CLIENT_ID"],
    client_secret=os.environ["SPOTIFY_CLIENT_SECRET"],
    redirect_uri=os.environ["SPOTIFY_REDIRECT_URI"],
    scope=SPOTIFY_SCOPES,
    open_browser=False
))

# Test the connection
user = sp.current_user()
print(f"Spotify connected successfully")
print(f"Logged in as: {user['display_name']}")

## LLM Setup

This section configures the LLM that powers the agent's reasoning

In [0]:
SYSTEM_PROMPT = """
You are VibeML, a personalized Spotify playlist agent. Your job is to help the user build a playlist that fits exactly what they are feeling or doing right now.

You have access to the following tools:
- get_user_profile: Pull the user's liked songs, playlists, and listening history from Spotify
- get_currently_playing: Fetch the track the user is currently listening to
- search_catalog: Verify a song exists and is available on Spotify
- search_songs: Search for songs by audio feature ranges like energy, valence, tempo, and danceability
- trend_filter: Filter songs by release era or artist popularity
- vibe_mapping: Translate the user's mood or activity description into target audio feature ranges
- feedback_refinement: Adjust audio feature ranges based on the user's tester song feedback
- save_playlist: Save the final playlist to the user's Spotify account
- constraint_filter: Filter songs based on user preferences or limits like explicit content or duration
- deduplication: Prevent the playlist from becoming repetitive by ensuring artist diversity
- liked_songs_matcher: Find the user's liked songs that match the target vibe and mix them into the playlist

Follow these steps when helping a user:
1. Pull the user's profile to understand their taste
2. Ask about their current mood, activity, and desired vibe
3. Use vibe_mapping to translate their input into audio feature ranges
4. Use liked_songs_matcher to find liked songs that match the vibe
5. Search for matching songs using search_songs
6. Apply constraint_filter based on any user preferences
7. Use deduplication to ensure playlist diversity
8. Mix liked songs with dataset results
9. Present 3 tester songs and collect feedback
10. Use feedback_refinement to adjust if needed
11. Verify songs on Spotify using search_catalog
12. Build and save the final playlist

Only help with music playlist requests. Politely decline anything outside of that scope.
"""

print("System prompt loaded successfully")

In [0]:
from openai import OpenAI
import mlflow

# LLM model configuration
LLM_MODEL = "databricks-gpt-oss-120b" # Can be changed, just an example

# Get the workspace URL and token automatically from the Databricks environment
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
DATABRICKS_HOST = spark.conf.get("spark.databricks.workspaceUrl")

# Set up the Databricks OpenAI compatible client
client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"https://{DATABRICKS_HOST}/serving-endpoints"
)

# Test that the model endpoint is accessible
def test_llm_setup():
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "user", "content": "Say hello in one sentence."}
        ]
    )
    # Extract text content only, skipping reasoning blocks
    content = response.choices[0].message.content
    if isinstance(content, list):
        text = " ".join([block["text"] for block in content if block.get("type") == "text"])
    else:
        text = content
    
    print("LLM setup successful")
    print(f"Model: {LLM_MODEL}")
    print(f"Response: {text}")

test_llm_setup()

## Tool Definitions

The following functions represent the tools the agent can call during its reasoning process.

### Tool 1 - Get User Profile: get_user_profile
- Pulls the user's liked songs, saved playlists, and listening history from the Spotify API. Also extracts the user's top artists and infers their preferred genres by cross referencing their listening activity against the Kaggle dataset. This profile is passed into other tools to personalize recommendations without the user having to explicitly state their preferences.


In [0]:
def get_user_profile(sp):
    """
    Pulls the user's liked songs, saved playlists, listening history,
    and infers genre preferences from their listening activity.
    
    Args:
        sp: Authenticated Spotipy client
    Returns:
        dict: User's liked songs, playlists, recently played, 
              top artists, and inferred genre preferences
    """
    # Get liked songs (up to 50)
    liked_songs = sp.current_user_saved_tracks(limit=50)
    liked_tracks = [
        {
            "track_name": item["track"]["name"],
            "artist": item["track"]["artists"][0]["name"],
            "track_id": item["track"]["id"]
        }
        for item in liked_songs["items"]
    ]

    # Get user playlists (up to 20)
    playlists = sp.current_user_playlists(limit=20)
    user_playlists = [
        {
            "playlist_name": playlist["name"],
            "track_count": playlist.get("tracks", {}).get("total", 0)
        }
        for playlist in playlists["items"]
    ]

    # Get recently played tracks (up to 50)
    recently_played = sp.current_user_recently_played(limit=50)
    recent_tracks = [
        {
            "track_name": item["track"]["name"],
            "artist": item["track"]["artists"][0]["name"],
            "track_id": item["track"]["id"]
        }
        for item in recently_played["items"]
    ]

    # Extract top artists from liked songs and recently played
    from collections import Counter
    all_artists = (
        [track["artist"] for track in liked_tracks] +
        [track["artist"] for track in recent_tracks]
    )
    top_artists = [artist for artist, count in Counter(all_artists).most_common(10)]

    # Infer genre preferences by looking up top artists in the Kaggle dataset
    df = spark.table("main.vibeml.vibeml_tracks_processed").toPandas()
    artist_genres = {}
    for artist in top_artists:
        artist_tracks = df[df["artists"].str.contains(artist, case=False, na=False)]
        if len(artist_tracks) > 0:
            top_genre = artist_tracks["track_genre"].value_counts().idxmax()
            artist_genres[artist] = top_genre

    # Count genre frequency to get top preferred genres
    genre_counts = Counter(artist_genres.values())
    preferred_genres = [genre for genre, count in genre_counts.most_common(5)]

    return {
        "liked_tracks": liked_tracks,
        "playlists": user_playlists,
        "recently_played": recent_tracks,
        "top_artists": top_artists,
        "preferred_genres": preferred_genres
    }

In [0]:
# Test the updated tool
profile = get_user_profile(sp)
print(f"Liked songs pulled: {len(profile['liked_tracks'])}")
print(f"Playlists pulled: {len(profile['playlists'])}")
print(f"Recently played pulled: {len(profile['recently_played'])}")
print(f"\nTop artists: {profile['top_artists']}")
print(f"Preferred genres: {profile['preferred_genres']}")

### Tool 2 - Get Current Song Playing: get_currently_playing
- Fetches the track the user is currently listening to from the Spotify API.


In [0]:
def get_currently_playing(sp):
    """
    Fetches the track the user is currently listening to from the Spotify API.
    
    Args:
        sp: Authenticated Spotipy client
    Returns:
        dict: Currently playing track info
    """
    pass

### Tool 3 - Search Catalog: search_catalog
- Searches the Spotify catalog by track name or artist to verify that a recommended song exists on the platform and is available to the user.

In [0]:
def search_catalog(sp, track_name=None, artist_name=None):
    """
    Searches the Spotify catalog to verify a track exists and is available.
    
    Args:
        sp: Authenticated Spotipy client
        track_name (str): Name of the track to search for
        artist_name (str): Name of the artist
    Returns:
        dict: Track info if found, None if not found
    """
    query = ""
    if track_name:
        query += f"track:{track_name} "
    if artist_name:
        query += f"artist:{artist_name}"
    
    query = query.strip()
    
    if not query:
        print("No search parameters provided")
        return None
    
    results = sp.search(q=query, limit=1, type="track")
    
    if results["tracks"]["items"]:
        track = results["tracks"]["items"][0]
        return {
            "track_name": track["name"],
            "artist": track["artists"][0]["name"],
            "track_id": track["id"],
            "album": track["album"]["name"],
            "available": True
        }
    else:
        return None

In [0]:
# Test the tool
result = search_catalog(sp, track_name="Rap God", artist_name="Eminem")
if result:
    print(f"Track found: {result['track_name']} by {result['artist']}")
    print(f"Album: {result['album']}")
    print(f"Track ID: {result['track_id']}")
else:
    print("Track not found")

### Tool 4 - Song Search: search_songs
- QSearches the first Kaggle dataset by audio feature ranges like energy, valence, danceability, and tempo. If the user specifies a genre it filters by that. If not, it automatically pulls from the user's preferred genres from their profile so the results already feel personal without the user's help. It also uses the second Kaggle dataset to rank results by artist popularity. Familiar artists from the user's liked songs and recently played always show up first.


In [0]:
def search_songs(energy=None, valence=None, tempo=None, 
                 danceability=None, speechiness=None, 
                 instrumentalness=None, genre=None,
                 limit=20, profile=None, min_results=15,
                 df_tracks=None, df_popularity=None):
    """
    Queries the Spotify Tracks Dataset (114,000 tracks across 125 genres) by audio 
    feature ranges to find matching tracks. If results are too few, automatically 
    widens the audio feature ranges before giving up. If a user profile is provided, 
    results are personalized by prioritizing familiar artists and preferred genres.
    Artist popularity from the Spotify Global Music Dataset is used to further rank results.
    
    Args:
        energy (tuple): Min and max energy range e.g. (0.7, 1.0)
        valence (tuple): Min and max valence range
        tempo (tuple): Min and max tempo range
        danceability (tuple): Min and max danceability range
        speechiness (tuple): Min and max speechiness range
        instrumentalness (tuple): Min and max instrumentalness range
        genre (str): Target genre to filter by
        limit (int): Max number of tracks to return
        profile (dict): User profile from get_user_profile tool
        min_results (int): Minimum acceptable results before widening filters
        df_tracks (DataFrame): Pre-loaded first Kaggle dataset
        df_popularity (DataFrame): Pre-loaded second Kaggle dataset for artist popularity
    Returns:
        DataFrame: Tracks matching the specified audio feature ranges
    """
    def apply_filters(df, energy, valence, tempo, danceability, speechiness, instrumentalness, genre, profile):
        if energy:
            df = df[(df["energy"] >= energy[0]) & (df["energy"] <= energy[1])]
        if valence:
            df = df[(df["valence"] >= valence[0]) & (df["valence"] <= valence[1])]
        if tempo:
            df = df[(df["tempo"] >= tempo[0]) & (df["tempo"] <= tempo[1])]
        if danceability:
            df = df[(df["danceability"] >= danceability[0]) & (df["danceability"] <= danceability[1])]
        if speechiness:
            df = df[(df["speechiness"] >= speechiness[0]) & (df["speechiness"] <= speechiness[1])]
        if instrumentalness:
            df = df[(df["instrumentalness"] >= instrumentalness[0]) & (df["instrumentalness"] <= instrumentalness[1])]
        if genre and genre != "any":
            df = df[df["track_genre"].str.lower() == genre.lower()]
        elif profile and profile.get("preferred_genres"):
            preferred = profile["preferred_genres"]
            df = df[df["track_genre"].str.lower().isin([g.lower() for g in preferred])]
        return df

    def widen_ranges(energy, valence, danceability, speechiness, instrumentalness, amount=0.1):
        def widen(r):
            if r is None:
                return None
            return (max(0.0, r[0] - amount), min(1.0, r[1] + amount))
        return (
            widen(energy),
            widen(valence),
            widen(danceability),
            widen(speechiness),
            widen(instrumentalness)
        )

    # Use pre-loaded datasets if provided otherwise load them
    df_full = df_tracks.copy() if df_tracks is not None else spark.table("main.vibeml.vibeml_tracks_processed").toPandas()
    df2 = df_popularity.copy() if df_popularity is not None else spark.table("main.vibeml.spotify_data_clean_raw").toPandas()

    # First attempt with original ranges
    df = apply_filters(df_full.copy(), energy, valence, tempo, danceability, speechiness, instrumentalness, genre, profile)
    df = df.drop_duplicates(subset=["track_name"]).reset_index(drop=True)

    # If results are too few widen ranges up to 3 times
    attempts = 0
    while len(df) < min_results and attempts < 3:
        attempts += 1
        print(f"Not enough results ({len(df)}), widening audio feature ranges (attempt {attempts})...")
        energy, valence, danceability, speechiness, instrumentalness = widen_ranges(
            energy, valence, danceability, speechiness, instrumentalness, amount=0.1 * attempts
        )
        ddf = apply_filters(df_full.copy(), energy, valence, tempo, danceability, speechiness, instrumentalness, genre, profile)
        df = df.drop_duplicates(subset=["track_name"]).reset_index(drop=True)

    if len(df) == 0:
        print("No results found even after widening. Returning top songs from genre only.")
        df = df_full[df_full["track_genre"].str.lower() == genre.lower()] if genre and genre != "any" else df_full
        df = df.drop_duplicates(subset=["track_name"]).reset_index(drop=True)

    # Load artist popularity from second dataset
    popularity_lookup = df2.groupby("artist_name")["artist_popularity"].mean().to_dict()
    df["artist_popularity_score"] = df["artists"].apply(
        lambda x: max([popularity_lookup.get(a.strip(), 0) for a in x.split(";")])
    )

    # Personalization
    if profile:
        liked_artists = set([track["artist"] for track in profile.get("liked_tracks", [])])
        recent_artists = set([track["artist"] for track in profile.get("recently_played", [])])
        familiar_artists = liked_artists.union(recent_artists)
        preferred_genres = set(profile.get("preferred_genres", []))

        df["is_familiar"] = df["artists"].apply(
            lambda x: any(artist in x for artist in familiar_artists)
        )
        df["is_preferred_genre"] = df["track_genre"].str.lower().isin(
            [g.lower() for g in preferred_genres]
        )

        df = df.sort_values(
            by=["is_familiar", "is_preferred_genre", "artist_popularity_score"],
            ascending=[False, False, False]
        )

        familiar_limit = min(len(df[df["is_familiar"]]), int(limit * 0.3))
        new_limit = limit - familiar_limit

        familiar_sample = df[df["is_familiar"]].head(familiar_limit)
        new_sample = df[~df["is_familiar"]].head(new_limit)

        df = pd.concat([familiar_sample, new_sample]).reset_index(drop=True)
    else:
        df = df.sort_values("artist_popularity_score", ascending=False).head(limit).reset_index(drop=True)

    return df

In [0]:
df_tracks = spark.table("main.vibeml.vibeml_tracks_processed").toPandas()
df_popularity = spark.table("main.vibeml.spotify_data_clean_raw").toPandas()
profile = get_user_profile(sp)

results = search_songs(
    energy=(0.8, 1.0),
    danceability=(0.7, 1.0),
    genre="afrobeat",
    limit=20,
    profile=profile,
    df_tracks=df_tracks,
    df_popularity=df_popularity
)
print(f"Found {len(results)} tracks")
print(results[["track_name", "artists", "energy", "danceability", "is_familiar"]].head(10))

### Tool 5 - Trend Filter: trend_filter
- Queries the second Kaggle dataset to narrow recommendations by release era or artist popularity.


In [0]:
def trend_filter(era=None, popularity=None):
    """
    Queries the Spotify Global Music Dataset (2009-2025) to narrow recommendations 
    by release era or artist popularity.
    
    Args:
        era (tuple): Min and max release year range e.g. (2009, 2015)
        popularity (tuple): Min and max artist popularity range
    Returns:
        DataFrame: Tracks matching the specified trend filters
    """
    pass

### Tool 6 - Vibe Mapping: vibe_mapping
- Takes the user's natural language description of their mood or activity and passes it to the LLM, which interprets the input and outputs target audio feature ranges and a genre. If the user's description is too vague, the tool asks a follow up question to better understand what they want before mapping the vibe.

In [0]:
def vibe_mapping(user_input, profile=None):
    """
    Takes the user's natural language description of their mood or activity and passes it to the LLM,
    which interprets the input and outputs a set of target audio feature ranges and a target genre.
    If the description is too vague, the LLM asks a follow up question first.
    
    Args:
        user_input (str): User's natural language description of their mood or activity
        profile (dict): User profile to inform genre suggestions
    Returns:
        dict: Target audio feature ranges and genre derived from the user's input
    """
    # Build genre context from user profile if available
    genre_context = ""
    if profile and profile.get("preferred_genres"):
        genre_context = f"The user typically listens to these genres: {', '.join(profile['preferred_genres'])}. Use this to inform your genre suggestion if the user does not specify one."

    prompt = f"""
    You are a music expert helping build a personalized Spotify playlist.
    
    {genre_context}
    
    Based on the user's description, first decide if you have enough information to map their vibe.
    If the description is too vague to confidently pick audio features and a genre, return a JSON 
    object with a single "follow_up" key containing one short clarifying question.
    
    If you have enough information, return a JSON object with target audio feature ranges and a genre.
    
    Each feature should have a "min" and "max" value between 0.0 and 1.0, except tempo which is in BPM.
    
    For genre, pick the single most relevant genre from this list:
    acoustic, afrobeat, alt-rock, alternative, ambient, anime, black-metal, bluegrass, blues, 
    bossanova, brazil, breakbeat, british, cantopop, chicago-house, children, chill, classical, 
    club, comedy, country, dance, dancehall, death-metal, deep-house, detroit-techno, disco, 
    disney, drum-and-bass, dub, dubstep, edm, electro, electronic, emo, folk, forro, french, 
    funk, garage, german, gospel, goth, grindcore, groove, grunge, guitar, happy, hard-rock, 
    hardcore, hardstyle, heavy-metal, hip-hop, holidays, honky-tonk, house, idm, indian, indie, 
    indie-pop, industrial, iranian, j-dance, j-idol, j-pop, j-rock, jazz, k-pop, kids, latin, 
    latino, malay, mandopop, metal, metal-misc, metalcore, minimal-techno, movies, mpb, new-age, 
    new-release, opera, pagode, party, philippines-opm, piano, pop, pop-film, power-pop, 
    progressive-house, psych-rock, punk, punk-rock, r-n-b, rainy-day, reggae, reggaeton, 
    road-trip, rock, rock-n-roll, rockabilly, romance, sad, samba, sertanejo, show-tunes, 
    singer-songwriter, ska, sleep, songwriter, soul, soundtracks, spanish, study, summer, 
    swedish, synth-pop, tango, techno, trance, trip-hop, turkish, work-out, world-music
    
    If no specific genre fits, return "any" for the genre field.
    
    User description: {user_input}
    
    Return ONLY a valid JSON object with no explanation or extra text. 
    
    If you need clarification:
    {{"follow_up": "your clarifying question here"}}
    
    If you have enough information:
    {{
        "energy": {{"min": 0.7, "max": 1.0}},
        "valence": {{"min": 0.5, "max": 1.0}},
        "danceability": {{"min": 0.7, "max": 1.0}},
        "speechiness": {{"min": 0.0, "max": 0.1}},
        "instrumentalness": {{"min": 0.0, "max": 0.3}},
        "tempo": {{"min": 120, "max": 180}},
        "genre": "most relevant genre from the list"
    }}
    """
    
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    
    # Handle both string and list response formats
    content = response.choices[0].message.content
    if isinstance(content, list):
        raw = " ".join([block["text"] for block in content if block.get("type") == "text"])
    else:
        raw = content
    
    # Clean up any markdown formatting if present
    raw = raw.replace("```json", "").replace("```", "").strip()
    result = json.loads(raw)
    
    # If the LLM needs clarification, ask the user and re-run
    if "follow_up" in result:
        print(f"\nAgent: {result['follow_up']}")
        follow_up_input = input("You: ").strip()
        combined_input = f"{user_input}. {follow_up_input}"
        return vibe_mapping(combined_input, profile)
    
    return result

In [0]:
# Test with a clear description
print("Test 1: Clear description")
result1 = vibe_mapping("I am heading to the gym and want something high energy hip hop", profile=profile)
print(json.dumps(result1, indent=2))

# Test with a vague description
print("\nTest 2: Vague description")
result2 = vibe_mapping("I want something good", profile=profile)
print(json.dumps(result2, indent=2))

### Tool 7 - Feedback Refinement: feedback_refinement
- Adjusts audio feature target ranges based on the user's tester song feedback.



In [0]:
def feedback_refinement(accepted_tracks, rejected_tracks, current_ranges, df):
    """
    Adjusts audio feature target ranges based on the user's tester song feedback.
    
    Args:
        accepted_tracks (list): Track names the user accepted
        rejected_tracks (list): Track names the user rejected
        current_ranges (dict): Current audio feature target ranges
        df (DataFrame): The dataset used for the search
    Returns:
        dict: Updated audio feature ranges
    """
    updated_ranges = current_ranges.copy()

    # Get audio features of rejected tracks and tighten ranges away from them
    if rejected_tracks:
        rejected_df = df[df["track_name"].isin(rejected_tracks)]
        for feature in ["energy", "valence", "danceability", "tempo"]:
            if feature in rejected_df.columns and feature in updated_ranges:
                rejected_mean = rejected_df[feature].mean()
                current_min = updated_ranges[feature]["min"]
                current_max = updated_ranges[feature]["max"]
                midpoint = (current_min + current_max) / 2
                # Push range away from rejected mean
                if rejected_mean < midpoint:
                    updated_ranges[feature]["min"] = min(current_max, current_min + 0.1)
                else:
                    updated_ranges[feature]["max"] = max(current_min, current_max - 0.1)

    return updated_ranges

In [0]:
# Test the tool
current_ranges = {
    "energy": {"min": 0.8, "max": 1.0},
    "valence": {"min": 0.5, "max": 1.0},
    "danceability": {"min": 0.7, "max": 1.0},
    "tempo": {"min": 120, "max": 150}
}

df = spark.table("main.vibeml.vibeml_tracks_processed").toPandas()

accepted = ["Yakuza"]
rejected = ["Touch Me"]

updated = feedback_refinement(accepted, rejected, current_ranges, df)
print("Updated ranges after feedback:")
print(json.dumps(updated, indent=2))

### Tool 8 - Save Playlist: save_playlist
- Saves the final list of tracks as a new playlist to the user's Spotify account.

In [0]:
def save_playlist(sp, track_ids, playlist_name):
    """
    Saves the final list of tracks as a new playlist to the user's Spotify account.
    
    Args:
        sp: Authenticated Spotipy client
        track_ids (list): List of Spotify track IDs to add to the playlist
        playlist_name (str): Name of the playlist to create
    Returns:
        str: URL of the created playlist
    """
    pass

### Tool 9: Constraint Filter: constraint_filter

Filters songs from the dataset based on the user's specific preferences or limits such as explicit content, song duration, or popularity.

In [0]:
def constraint_filter(df, explicit=None, max_duration_ms=None, min_popularity=None):
    """
    Filters songs based on the user's specific preferences or limits.
    If applying the popularity floor would empty the result set, the floor
    is skipped rather than returning zero songs.
    
    Args:
        df (DataFrame): The dataset to filter
        explicit (bool): If False, exclude explicit tracks. If True, include only explicit tracks. If None, no filter.
        max_duration_ms (int): Maximum song duration in milliseconds
        min_popularity (int): Minimum popularity score between 0 and 100
    Returns:
        DataFrame: Filtered tracks based on the specified constraints
    """
    filtered = df.copy()

    if explicit is not None:
        filtered = filtered[filtered["explicit"] == explicit]

    if max_duration_ms is not None:
        filtered = filtered[filtered["duration_ms"] <= max_duration_ms]

    if min_popularity is not None:
        popularity_filtered = filtered[filtered["popularity"] >= min_popularity]
        if len(popularity_filtered) > 0:
            filtered = popularity_filtered
        else:
            print(f"Popularity floor of {min_popularity} would return 0 songs, skipping it")

    return filtered.reset_index(drop=True)

constraint_filter = mlflow.trace(name="constraint_filter")(constraint_filter)

print("constraint_filter fixed and re-wrapped")

In [0]:
# Test the tool
df = spark.table("main.vibeml.vibeml_tracks_processed").toPandas()

filtered = constraint_filter(df, explicit=False, max_duration_ms=300000, min_popularity=50)
print(f"Total tracks before filtering: {len(df)}")
print(f"Total tracks after filtering: {len(filtered)}")
print(filtered[["track_name", "artists", "explicit", "duration_ms", "popularity"]].head(5))

### Tool 10 - Diversity Tool: deduplication

Prevents the playlist from becoming repetitive by ensuring no artist appears more than once and removing duplicate tracks.

In [0]:
def deduplication(df, max_per_artist=1):
    """
    Prevents the playlist from becoming repetitive by limiting how many times
    an artist can appear and removing duplicate tracks.
    
    Args:
        df (DataFrame): The dataset to deduplicate
        max_per_artist (int): Maximum number of tracks allowed per artist
    Returns:
        DataFrame: Deduplicated tracks with artist diversity enforced
    """
    # Remove duplicate track names
    df = df.drop_duplicates(subset=["track_name"]).reset_index(drop=True)
    
    # Limit tracks per artist
    df = df.groupby("artists").head(max_per_artist).reset_index(drop=True)
    
    return df

In [0]:
# Test the tool
df = spark.table("main.vibeml.vibeml_tracks_processed").toPandas()

# First run a song search to get a subset
results = search_songs(energy=(0.8, 1.0), danceability=(0.7, 1.0), limit=50)
print(f"Tracks before deduplication: {len(results)}")
print(f"Unique artists before: {results['artists'].nunique()}")

deduped = deduplication(results, max_per_artist=1)
print(f"Tracks after deduplication: {len(deduped)}")
print(f"Unique artists after: {deduped['artists'].nunique()}")
print(deduped[["track_name", "artists"]].head(10))

### Tool 11: Liked Songs Matcher Tool

Looks up the user's liked songs in the Kaggle dataset and filters them by the target audio feature ranges to find ones that match the current vibe. This ensures the playlist includes songs the user already knows.

In [0]:
def liked_songs_matcher(profile, ranges, df):
    """
    Looks up the user's liked songs in the Kaggle dataset and filters
    them by the target audio feature ranges and genre (if specified)
    to find ones that match the vibe.
    
    Args:
        profile (dict): User profile from get_user_profile tool
        ranges (dict): Target audio feature ranges from vibe_mapping, may include "genre"
        df (DataFrame): The processed dataset
    Returns:
        DataFrame: Liked songs that match the target vibe
    """
    liked_tracks = profile.get("liked_tracks", [])
    liked_names = [track["track_name"].lower() for track in liked_tracks]
    liked_artists = [track["artist"].lower() for track in liked_tracks]
    
    matched = df[
        df["track_name"].str.lower().isin(liked_names) |
        df["artists"].str.lower().isin(liked_artists)
    ].copy()
    
    if "energy" in ranges:
        matched = matched[
            (matched["energy"] >= ranges["energy"]["min"]) & 
            (matched["energy"] <= ranges["energy"]["max"])
        ]
    if "valence" in ranges:
        matched = matched[
            (matched["valence"] >= ranges["valence"]["min"]) & 
            (matched["valence"] <= ranges["valence"]["max"])
        ]
    if "danceability" in ranges:
        matched = matched[
            (matched["danceability"] >= ranges["danceability"]["min"]) & 
            (matched["danceability"] <= ranges["danceability"]["max"])
        ]
    
    # Filter by genre too, if one was specified
    genre = ranges.get("genre")
    if genre and genre != "any":
        matched = matched[matched["track_genre"].str.lower() == genre.lower()]
    
    return matched.reset_index(drop=True)

# Re-wrap with tracing since this is a fresh function definition
liked_songs_matcher = mlflow.trace(name="liked_songs_matcher")(liked_songs_matcher)

print("liked_songs_matcher compiled")

In [0]:
# Test the tool
profile = get_user_profile(sp)
df = spark.table("main.vibeml.vibeml_tracks_processed").toPandas()
ranges = vibe_mapping("I am heading to the gym and want something high energy")

liked_matches = liked_songs_matcher(profile, ranges, df)
print(f"Liked songs that match the vibe: {len(liked_matches)}")
if len(liked_matches) > 0:
    print(liked_matches[["track_name", "artists", "energy", "danceability"]].head(5))
else:
    print("No liked songs matched the target vibe in the dataset")

## Agent Definition

This section defines the agent's system prompt, registers the tools, and sets up the LLM. The agent uses a ReAct style reasoning pattern, that is it reasons over the user's input, decides which tools to call, observes the results, and reasons again until it has enough information to build the final playlist.

In [0]:
# Tool list - all implemented tools
TOOLS = [
    get_user_profile,
    get_currently_playing,
    search_catalog,
    search_songs,
    trend_filter,
    vibe_mapping,
    feedback_refinement,
    save_playlist,
    constraint_filter,
    deduplication,
    liked_songs_matcher
]

print(f"Agent definition loaded successfully")
print(f"Total tools registered: {len(TOOLS)}")

## LLM-Driven Conversational Agent

The agent is driven by the LLM which reads the user's input and decides which tools to call and what to say next. The conversation continues until the user is satisfied with the playlist.

In [0]:
# Define tools in a format the LLM can reason about
TOOL_DEFINITIONS = {
    "vibe_mapping": {
        "description": "Translates the user's mood or activity description into audio feature ranges and a genre",
        "when_to_use": "When the user describes what they want to listen to"
    },
    "search_songs": {
        "description": "Searches the dataset for songs matching audio feature ranges and genre",
        "when_to_use": "After vibe mapping to find matching tracks"
    },
    "liked_songs_matcher": {
        "description": "Finds songs from the user's liked songs that match the target vibe",
        "when_to_use": "After vibe mapping to personalize results with familiar songs"
    },
    "constraint_filter": {
        "description": "Filters songs by explicit content, duration, or popularity",
        "when_to_use": "When the user mentions preferences like no explicit songs or popular songs only"
    },
    "deduplication": {
        "description": "Removes duplicate artists and tracks from the playlist",
        "when_to_use": "Before presenting tester songs to ensure diversity"
    },
    "search_catalog": {
        "description": "Verifies a song exists on Spotify",
        "when_to_use": "Before finalizing the playlist to confirm all songs are available"
    },
    "feedback_refinement": {
        "description": "Adjusts audio feature ranges based on accepted and rejected tester songs",
        "when_to_use": "When the user rejects one or more tester songs"
    }
}

def build_tool_context():
    """Builds a string description of available tools for the LLM system prompt."""
    tool_text = ""
    for name, info in TOOL_DEFINITIONS.items():
        tool_text += f"- {name}: {info['description']}\n"
    return tool_text

print("Tool definitions loaded")
print(build_tool_context())

### Conversational Agent Loop

The main agent loop driven by the LLM. The LLM reads every user message, decides which tools to call, interprets the results, and responds naturally. The conversation continues until the user is satisfied with the playlist.

In [0]:
# List of genres that exist in the dataset
VALID_GENRES = [
    "acoustic", "afrobeat", "alt-rock", "alternative", "ambient", "anime", 
    "black-metal", "bluegrass", "blues", "bossanova", "brazil", "breakbeat", 
    "british", "cantopop", "chicago-house", "children", "chill", "classical", 
    "club", "comedy", "country", "dance", "dancehall", "death-metal", "deep-house", 
    "detroit-techno", "disco", "disney", "drum-and-bass", "dub", "dubstep", "edm", 
    "electro", "electronic", "emo", "folk", "forro", "french", "funk", "garage", 
    "german", "gospel", "goth", "grindcore", "groove", "grunge", "guitar", "happy", 
    "hard-rock", "hardcore", "hardstyle", "heavy-metal", "hip-hop", "holidays", 
    "honky-tonk", "house", "idm", "indian", "indie", "indie-pop", "industrial", 
    "iranian", "j-dance", "j-idol", "j-pop", "j-rock", "jazz", "k-pop", "kids", 
    "latin", "latino", "malay", "mandopop", "metal", "metal-misc", "metalcore", 
    "minimal-techno", "movies", "mpb", "new-age", "new-release", "opera", "pagode", 
    "party", "philippines-opm", "piano", "pop", "pop-film", "power-pop", 
    "progressive-house", "psych-rock", "punk", "punk-rock", "r-n-b", "rainy-day", 
    "reggae", "reggaeton", "road-trip", "rock", "rock-n-roll", "rockabilly", 
    "romance", "sad", "samba", "sertanejo", "show-tunes", "singer-songwriter", 
    "ska", "sleep", "songwriter", "soul", "soundtracks", "spanish", "study", 
    "summer", "swedish", "synth-pop", "tango", "techno", "trance", "trip-hop", 
    "turkish", "work-out", "world-music"
]

def extract_explicit_genre(user_input):
    """
    Checks if the user explicitly mentioned a valid genre in their input.
    If found, returns the genre. Otherwise returns None.
    
    Args:
        user_input (str): The user's input
    Returns:
        str or None: The detected genre or None
    """
    user_input_lower = user_input.lower()
    for genre in VALID_GENRES:
        if genre in user_input_lower or genre.replace("-", " ") in user_input_lower:
            return genre
    return None

# Test it
print(extract_explicit_genre("I want fast afrobeat music"))
print(extract_explicit_genre("give me something chill"))
print(extract_explicit_genre("I want hip hop workout songs"))

In [0]:
def run_agent():
    """
    LLM-driven conversational agent that builds a personalized playlist
    through natural conversation. The LLM decides which tools to call
    and what to say next based on the user's input.
    """
    print("=" * 50)
    print("Welcome to VibeML!")
    print("=" * 50)

    # Step 1: Pull user profile once at the start
    print("\nPulling your Spotify profile...")
    profile = get_user_profile(sp)
    df = spark.table("main.vibeml.vibeml_tracks_processed").toPandas()
    print(f"Got your profile. Top genres: {profile['preferred_genres']}")
    print(f"Top artists: {profile['top_artists'][:5]}")

    # Initialize conversation history
    conversation_history = [
        {
            "role": "system",
            "content": f"""You are VibeML, a personalized Spotify playlist agent. 
Your job is to have a natural conversation with the user to build them the perfect playlist for this moment.

You know the following about this user already:
- Top artists: {', '.join(profile['top_artists'])}
- Preferred genres: {', '.join(profile['preferred_genres'])}

You have access to these tools:
{build_tool_context()}

Your conversation flow should be:
1. Greet the user warmly and ask what kind of playlist they want right now
2. When you have enough info, tell the user you are finding songs for them
3. Present 3 tester songs with a one sentence explanation for each one
4. Ask which songs they like or dislike
5. Refine based on feedback if needed
6. When the user is happy, summarize the final playlist and confirm saving it
7. Keep the conversation natural and friendly throughout

Important rules:
- Never expose technical terms like valence, danceability, or audio features to the user
- Always explain song picks in plain language
- If the user says something off topic, gently redirect them back to building their playlist
- Only help with music playlist requests
- Keep responses concise and conversational"""
        }
    ]

    # Agent state
    current_ranges = None
    current_results = None
    tester_songs = None
    final_playlist = None
    playlist_saved = False

    # Start the conversation
    agent_intro = client.chat.completions.create(
        model=LLM_MODEL,
        messages=conversation_history
    )

    content = agent_intro.choices[0].message.content
    if isinstance(content, list):
        intro_text = " ".join([block["text"] for block in content if block.get("type") == "text"])
    else:
        intro_text = content

    print(f"\nVibeML: {intro_text}")
    conversation_history.append({"role": "assistant", "content": intro_text})

    # Main conversation loop
    while not playlist_saved:
        user_input = input("\nYou: ").strip()

        if not user_input:
            continue

        # Add user message to history
        conversation_history.append({"role": "user", "content": user_input})

        # Check if the user wants to end the conversation entirely
        exit_phrases = ["i'm done", "im done", "stop", "exit", "quit", "that's all", 
                        "thats all", "i'm good", "im good", "no thanks", "goodbye", 
                        "bye", "nothing else", "that's it", "thats it"]
        if any(phrase in user_input.lower() for phrase in exit_phrases):
            farewell_response = client.chat.completions.create(
                model=LLM_MODEL,
                messages=conversation_history
            )
            content = farewell_response.choices[0].message.content
            if isinstance(content, list):
                farewell_text = " ".join([block["text"] for block in content if block.get("type") == "text"])
            else:
                farewell_text = content
            print(f"\nVibeML: {farewell_text}")
            playlist_saved = True
            break

        # Check if user wants to save or is done
        done_keywords = ["save it", "save the playlist", "yes save", "looks good save", 
                        "perfect save", "i'm happy", "im happy", "that's perfect", "save"]
        if any(keyword in user_input.lower() for keyword in done_keywords):
            if final_playlist:
                summary_prompt = f"""The user wants to save the playlist. 
                Summarize these {len(final_playlist)} songs in a warm friendly way, 
                mentioning the overall vibe and a few song highlights: 
                {[t['track_name'] + ' by ' + t['artist'] for t in final_playlist]}"""
                
                conversation_history.append({"role": "user", "content": summary_prompt})
                summary_response = client.chat.completions.create(
                    model=LLM_MODEL,
                    messages=conversation_history
                )
                content = summary_response.choices[0].message.content
                if isinstance(content, list):
                    summary_text = " ".join([block["text"] for block in content if block.get("type") == "text"])
                else:
                    summary_text = content

                print(f"\nVibeML: {summary_text}")

                # Ask if the user needs anything else
                anything_else_response = client.chat.completions.create(
                    model=LLM_MODEL,
                    messages=conversation_history + [
                        {"role": "user", "content": "Ask the user if they need anything else or if they are done."}
                    ]
                )
                content = anything_else_response.choices[0].message.content
                if isinstance(content, list):
                    anything_else_text = " ".join([block["text"] for block in content if block.get("type") == "text"])
                else:
                    anything_else_text = content

                print(f"\nVibeML: {anything_else_text}")
                conversation_history.append({"role": "assistant", "content": anything_else_text})

                # Wait for user response
                user_input = input("\nYou: ").strip()
                conversation_history.append({"role": "user", "content": user_input})

                done_phrases = ["no", "i'm good", "im good", "that's all", "thats all", "done", 
                                "goodbye", "bye", "nothing", "nope", "all good"]

                if any(phrase in user_input.lower() for phrase in done_phrases):
                    farewell_response = client.chat.completions.create(
                        model=LLM_MODEL,
                        messages=conversation_history
                    )
                    content = farewell_response.choices[0].message.content
                    if isinstance(content, list):
                        farewell_text = " ".join([block["text"] for block in content if block.get("type") == "text"])
                    else:
                        farewell_text = content
                    print(f"\nVibeML: {farewell_text}")
                    playlist_saved = True
                    break
                else:
                    # User wants something else, reset state for a new playlist
                    current_ranges = None
                    current_results = None
                    tester_songs = None
                    final_playlist = None
                    print(f"\nVibeML: Sure, let's build another one! What vibe are you going for?")

        # Decide what to do next based on conversation state
        if current_ranges is None:
            try:
                result = vibe_mapping(user_input, profile)
                
                if "follow_up" in result:
                    # LLM needs more info, ask the user
                    print(f"\nVibeML: {result['follow_up']}")
                    conversation_history.append({"role": "assistant", "content": result['follow_up']})
                    # Do not set current_ranges yet, wait for user response
                else:
                    current_ranges = result
                    
                    # Override genre if user explicitly mentioned one
                    explicit_genre = extract_explicit_genre(user_input)
                    if explicit_genre:
                        current_ranges["genre"] = explicit_genre
                        print(f"Genre override applied: {explicit_genre}")

                    # Search for songs
                    current_results = search_songs(
                        energy=(current_ranges["energy"]["min"], current_ranges["energy"]["max"]),
                        valence=(current_ranges["valence"]["min"], current_ranges["valence"]["max"]),
                        danceability=(current_ranges["danceability"]["min"], current_ranges["danceability"]["max"]),
                        speechiness=(current_ranges["speechiness"]["min"], current_ranges["speechiness"]["max"]),
                        instrumentalness=(current_ranges["instrumentalness"]["min"], current_ranges["instrumentalness"]["max"]),
                        genre=current_ranges.get("genre"),
                        limit=50,
                        profile=profile
                    )

                    # Apply filters
                    current_results = constraint_filter(current_results, explicit=False, min_popularity=30)
                    current_results = deduplication(current_results, max_per_artist=1)

                    # Mix in liked songs
                    liked_matches = liked_songs_matcher(profile, current_ranges, df)
                    if len(liked_matches) > 0:
                        liked_matches["is_familiar"] = True
                        current_results = pd.concat([liked_matches, current_results]).drop_duplicates(subset=["track_name"]).reset_index(drop=True)

                    # Pick tester songs
                    tester_songs = current_results.head(3)

                    # Ask LLM to present tester songs naturally with explanations
                    tester_info = []
                    for _, row in tester_songs.iterrows():
                        familiar_note = "from your liked songs" if row.get("is_familiar") else "a new discovery"
                        tester_info.append(f"{row['track_name']} by {row['artists']} ({familiar_note}, genre: {row['track_genre']})")

                    presentation_prompt = f"""Present these 3 tester songs to the user in a natural friendly way. 
                    For each song give a one sentence explanation of why it fits their vibe.
                    The user described wanting: {user_input}
                    Their preferred genres are: {profile['preferred_genres']}
                    
                    Songs:
                    1. {tester_info[0]}
                    2. {tester_info[1]}
                    3. {tester_info[2]}
                    
                    After presenting them ask which ones they like and which ones they do not like.
                    Make it clear they can accept all, reject all, or give mixed feedback."""

                    conversation_history.append({"role": "user", "content": presentation_prompt})
                    presentation_response = client.chat.completions.create(
                        model=LLM_MODEL,
                        messages=conversation_history
                    )
                    content = presentation_response.choices[0].message.content
                    if isinstance(content, list):
                        presentation_text = " ".join([block["text"] for block in content if block.get("type") == "text"])
                    else:
                        presentation_text = content

                    print(f"\nVibeML: {presentation_text}")
                    conversation_history.append({"role": "assistant", "content": presentation_text})

            except Exception as e:
                response = client.chat.completions.create(
                    model=LLM_MODEL,
                    messages=conversation_history
                )
                content = response.choices[0].message.content
                if isinstance(content, list):
                    response_text = " ".join([block["text"] for block in content if block.get("type") == "text"])
                else:
                    response_text = content
                print(f"\nVibeML: {response_text}")
                conversation_history.append({"role": "assistant", "content": response_text})

        else:
            # Parse feedback more intelligently using the LLM
            feedback_parse_prompt = f"""The user said: "{user_input}"
            
            The tester songs were:
            {[row['track_name'] for _, row in tester_songs.iterrows()] if tester_songs is not None else []}
            
            Classify the user's response into one of these categories and return ONLY a JSON object:
            - "accept_all": user likes all songs and wants to proceed
            - "reject_all": user dislikes all songs
            - "mixed": user likes some but not others
            - "off_topic": user is asking about something unrelated to music
            - "other": unclear or conversational response
            
            Also extract:
            - "accepted_songs": list of song names the user explicitly liked (empty list if none mentioned)
            - "rejected_songs": list of song names the user explicitly disliked (empty list if none mentioned)
            
            Return ONLY valid JSON like this:
            {{"category": "mixed", "accepted_songs": ["song1"], "rejected_songs": ["song2"]}}"""

            parse_response = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": feedback_parse_prompt}]
            )
            parse_content = parse_response.choices[0].message.content
            if isinstance(parse_content, list):
                parse_raw = " ".join([block["text"] for block in parse_content if block.get("type") == "text"])
            else:
                parse_raw = parse_content

            parse_raw = parse_raw.replace("```json", "").replace("```", "").strip()

            try:
                feedback_result = json.loads(parse_raw)
                category = feedback_result.get("category", "other")
                accepted_songs = feedback_result.get("accepted_songs", [])
                rejected_songs = feedback_result.get("rejected_songs", [])

                if category == "accept_all":
                    # Build final playlist
                    final_pool = current_results.head(20)
                    verified_tracks = []

                    print("\nVibeML: Building your playlist and verifying songs on Spotify...")

                    for _, row in final_pool.iterrows():
                        result = search_catalog(sp, track_name=row["track_name"],
                                              artist_name=row["artists"].split(";")[0])
                        if result:
                            verified_tracks.append(result)
                        else:
                            # Keep the song even if not found on Spotify catalog
                            verified_tracks.append({
                                "track_name": row["track_name"],
                                "artist": row["artists"].split(";")[0],
                                "track_id": None,
                                "album": None,
                                "available": False
                            })
                        if len(verified_tracks) >= 10:
                            break

                    final_playlist = verified_tracks

                    final_info = [f"{t['track_name']} by {t['artist']}" for t in final_playlist]
                    final_prompt = f"""Present the final playlist to the user in a warm friendly way.
                    Tell them it is ready and list all the songs.
                    Then ask if they want to save it to Spotify.
                    
                    Playlist:
                    {chr(10).join([f'{i+1}. {song}' for i, song in enumerate(final_info)])}"""

                    conversation_history.append({"role": "user", "content": final_prompt})
                    final_response = client.chat.completions.create(
                        model=LLM_MODEL,
                        messages=conversation_history
                    )
                    content = final_response.choices[0].message.content
                    if isinstance(content, list):
                        final_text = " ".join([block["text"] for block in content if block.get("type") == "text"])
                    else:
                        final_text = content

                    print(f"\nVibeML: {final_text}")
                    conversation_history.append({"role": "assistant", "content": final_text})

                elif category in ["reject_all", "mixed"]:
                    # Refine based on feedback
                    if tester_songs is not None:
                        if category == "reject_all":
                            rejected = tester_songs["track_name"].tolist()
                            accepted = []
                        else:
                            rejected = rejected_songs if rejected_songs else [
                                row["track_name"] for _, row in tester_songs.iterrows()
                                if row["track_name"] not in accepted_songs
                            ]
                            accepted = accepted_songs

                        current_ranges = feedback_refinement(accepted, rejected, current_ranges, current_results)

                        # Re-search with refined ranges
                        current_results = search_songs(
                            energy=(current_ranges["energy"]["min"], current_ranges["energy"]["max"]),
                            valence=(current_ranges["valence"]["min"], current_ranges["valence"]["max"]),
                            danceability=(current_ranges["danceability"]["min"], current_ranges["danceability"]["max"]),
                            genre=current_ranges.get("genre"),
                            limit=50,
                            profile=profile
                        )
                        current_results = constraint_filter(current_results, explicit=False, min_popularity=30)
                        current_results = deduplication(current_results, max_per_artist=1)

                        liked_matches = liked_songs_matcher(profile, current_ranges, df)
                        if len(liked_matches) > 0:
                            liked_matches["is_familiar"] = True
                            current_results = pd.concat([liked_matches, current_results]).drop_duplicates(subset=["track_name"]).reset_index(drop=True)

                        tester_songs = current_results.head(3)
                        tester_info = []
                        for _, row in tester_songs.iterrows():
                            familiar_note = "from your liked songs" if row.get("is_familiar") else "a new discovery"
                            tester_info.append(f"{row['track_name']} by {row['artists']} ({familiar_note}, genre: {row['track_genre']})")

                        refinement_prompt = f"""The user gave feedback: {user_input}
                        They liked: {accepted if accepted else 'none specifically mentioned'}
                        They did not like: {rejected}
                        
                        You have refined the recommendations. Present these new tester songs naturally
                        and explain why they better match what the user wants.
                        
                        New songs:
                        1. {tester_info[0]}
                        2. {tester_info[1]}
                        3. {tester_info[2]}
                        
                        Ask which ones they like or dislike."""

                        conversation_history.append({"role": "user", "content": refinement_prompt})
                        refinement_response = client.chat.completions.create(
                            model=LLM_MODEL,
                            messages=conversation_history
                        )
                        content = refinement_response.choices[0].message.content
                        if isinstance(content, list):
                            refinement_text = " ".join([block["text"] for block in content if block.get("type") == "text"])
                        else:
                            refinement_text = content

                        print(f"\nVibeML: {refinement_text}")
                        conversation_history.append({"role": "assistant", "content": refinement_text})

                elif category == "off_topic":
                    off_topic_response = client.chat.completions.create(
                        model=LLM_MODEL,
                        messages=conversation_history
                    )
                    content = off_topic_response.choices[0].message.content
                    if isinstance(content, list):
                        off_topic_text = " ".join([block["text"] for block in content if block.get("type") == "text"])
                    else:
                        off_topic_text = content
                    print(f"\nVibeML: {off_topic_text}")
                    conversation_history.append({"role": "assistant", "content": off_topic_text})

                else:
                    # Let LLM handle anything else
                    response = client.chat.completions.create(
                        model=LLM_MODEL,
                        messages=conversation_history
                    )
                    content = response.choices[0].message.content
                    if isinstance(content, list):
                        response_text = " ".join([block["text"] for block in content if block.get("type") == "text"])
                    else:
                        response_text = content
                    print(f"\nVibeML: {response_text}")
                    conversation_history.append({"role": "assistant", "content": response_text})

            except Exception as e:
                response = client.chat.completions.create(
                    model=LLM_MODEL,
                    messages=conversation_history
                )
                content = response.choices[0].message.content
                if isinstance(content, list):
                    response_text = " ".join([block["text"] for block in content if block.get("type") == "text"])
                else:
                    response_text = content
                print(f"\nVibeML: {response_text}")
                conversation_history.append({"role": "assistant", "content": response_text})

    return final_playlist

# Run the conversational agent
final_playlist = run_agent()